# 02 · Context, prompts, and a finite reading desk

**Question:** What does one model call actually receive?

You will inspect messages, budget a small context, and test a checkpoint that preserves an approval boundary. Lists, dictionaries, and functions are enough. We assemble inputs here; no language model runs in this notebook.

**Workshop checkpoint:** A and B. C is a short pair discussion. D is an extension.

Run cells from top to bottom with **Shift+Enter**. Before changing an experiment, predict its result. After editing, rerun that cell and the cells that depend on it. **Kernel → Restart Kernel and Run All Cells** checks that you have not relied on hidden state.

Exercises marked **YOUR TURN** deliberately use `None` until you answer. Their checks explain what is missing without stopping a first Run All. A completed exercise must print its PASS message. Worked versions are supplied separately to the instructor. All campus data is fictional.

In [ ]:
import math
import random
import json
from collections import Counter, defaultdict
import matplotlib.pyplot as plt

plt.rcParams.update({
    "figure.figsize": (9, 4), "figure.dpi": 110,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.prop_cycle": plt.cycler(color=["#236d4e", "#e0784c", "#708eaa"]),
    "axes.titlesize": 15, "axes.labelsize": 11
})
print("Ready. No network or model download is needed.")
import re
from copy import deepcopy

## A · Inspect the input

The application chooses what to supply. This includes instructions, the student request, selected history, and relevant evidence. A variable on your laptop is not automatically visible to the model.

We use a **toy tokenizer** that separates words and punctuation. Its counts belong only to this exercise. Real model tokenizers, chat wrappers, output limits, and reasoning-token accounting differ. Production budgets must use the chosen model/API's documented accounting.

In [ ]:
def toy_tokens(text):
    return re.findall(r"\w+|[^\w\s]", text, flags=re.UNICODE)

blocks = [
    {"id": "instructions", "role": "system", "content": "Help with course support. Use evidence. Ask before booking."},
    {"id": "request", "role": "user", "content": "Find recursion support after 16:00. I have not approved a booking."},
    {"id": "state", "role": "context", "content": "Goal: recursion after 16:00. Approved session: none. Booking: not attempted."},
    {"id": "guide-v2-s3", "role": "evidence", "content": "Week 3 recursion: identify the base case and trace factorial."},
    {"id": "history-dump", "role": "history", "content": "We discussed unrelated clubs and menus. " * 80},
]
def block_size(block):
    return len(toy_tokens(json.dumps(block, ensure_ascii=False)))
def used_tokens(selected, reply_reserve=40):
    # A deliberately simplified format: sum serialized blocks and a reserved reply.
    return sum(block_size(block) for block in selected) + reply_reserve

CONTEXT_LIMIT = 260
for block in blocks:
    print(f"{block['id']:16} {block_size(block):4} toy tokens")
print("Everything + reply reserve:", used_tokens(blocks))
print("Toy capacity:", CONTEXT_LIMIT)
print("\nActual message content:")
print(json.dumps(blocks[:2], indent=2))

## B · YOUR TURN: choose what survives

Keep the instructions, request, task state, and current guide passage. Omit the unrelated history dump. Set `chosen_ids` to a list of IDs. The test checks both required information and the budget.

**Predict:** Would keeping only the last message preserve the student's constraint and permission state?

In [ ]:
chosen_ids = None  # YOUR TURN: a list of block IDs
if chosen_ids is None:
    print("YOUR TURN B: select the blocks for the next call.")
else:
    known = {block["id"]: block for block in blocks}
    assert len(chosen_ids) == len(set(chosen_ids)), "Do not duplicate context."
    assert set(chosen_ids) <= set(known), "Use known IDs."
    assert {"instructions", "request", "state", "guide-v2-s3"} <= set(chosen_ids)
    selected = [known[key] for key in chosen_ids]
    assert used_tokens(selected) <= CONTEXT_LIMIT, "Context exceeds this toy capacity."
    print("PASS B:", used_tokens(selected), "toy tokens including reply reserve.")

In [ ]:
# Provided continuation so an unfinished exercise does not block the lesson.
selected_context = [block for block in blocks if block["id"] != "history-dump"]
sizes = [block_size(block) for block in selected_context] + [40]
labels = [block["id"] for block in selected_context] + ["reply reserve"]
fig, ax = plt.subplots(figsize=(10, 3))
left = 0
colors = ["#236d4e", "#7c9564", "#708eaa", "#e0784c", "#adb9b0"]
for label, size, color in zip(labels, sizes, colors):
    ax.barh(["Next call"], [size], left=left, label=label, color=color,
            hatch="///" if label == "reply reserve" else None)
    left += size
ax.axvline(CONTEXT_LIMIT, color="#b04432", linestyle="--", label="toy limit")
ax.set(xlabel="Toy token units", xlim=(0, CONTEXT_LIMIT + 35), title="Selected context fits")
ax.legend(ncol=3, loc="upper center", bbox_to_anchor=(0.5, -0.25))
plt.show()
assert used_tokens(selected_context) <= CONTEXT_LIMIT

naive_tail = blocks[-1:]
print("Naive last-block selection:", [b["id"] for b in naive_tail])
print("Lost required IDs:", {"instructions", "request", "state"} - {b["id"] for b in naive_tail})

## C · A prompt is a specification

Complete the request in your own words. Include the learner's prior knowledge, supplied notes, requested output, and what to do when evidence is absent. Then swap with a partner.

**A good check:** Can your partner point to an exercise whose answer would show understanding? A keyword check cannot tell whether a prompt or answer is actually good. We intentionally do not pretend to grade prompt quality with string matching.

In [ ]:
notes = "Recursion calls a function again with a smaller problem. A base case stops it."
your_prompt = """I know Java functions but have not used recursion.
Use the supplied notes to ..."""

print("SUPPLIED NOTES:", notes)
print("\nYOUR REQUEST:", your_prompt)
# YOUR TURN: edit your_prompt.
# Peer review: source boundary? appropriate learner? checkable exercise?
# This cell displays your input. It does not generate or grade an LLM answer.

## D · A checkpoint preserves the important state

After an attempted booking, the next call needs the original goal, the failed outcome, source references, and the precise approval scope. Summaries may omit details. Authoritative permissions belong in application state, outside a model-generated summary.

Compare the precise checkpoint with the vague note below. Nothing is written to disk by this example.

In [ ]:
checkpoint = {
    "goal": {"topic": "recursion", "after": "16:00"},
    "source_ids": ["guide-v2-s3"],
    "approved_session": "wed-1630",
    "booking_status": "failed_full",
    "next_step": "refresh availability and ask about alternatives",
}
vague_summary = {"goal": "help the student", "approved": True}

# JSON round-trip represents a stored checkpoint being loaded.
loaded_checkpoint = json.loads(json.dumps(checkpoint))
assert loaded_checkpoint == checkpoint
print(json.dumps(loaded_checkpoint, indent=2))
print("\nA model sees it only if the harness adds it to the next call.")

# Ordinary application check; this does not ask a model to infer permission.
def has_exact_approval(session_id, approval_record):
    return approval_record.get("approved_session") == session_id

assert has_exact_approval("wed-1630", loaded_checkpoint)
assert not has_exact_approval("fri-1700", loaded_checkpoint)
assert not has_exact_approval("fri-1700", vague_summary)
print("PASS: vague approval does not authorize a different session.")

### Extension challenge

Write `pack_context(required, candidates, limit, reserve)` that keeps every required block, adds candidate blocks in relevance order when they fit, and raises a clear error when the required input alone is too large. Test a zero-evidence case and a budget smaller than the required input.

Do not silently drop permissions or the user request to satisfy a budget. A production application may need to ask a clarifying question or start a new session.

## Explain what you observed

- How are context, learned parameters, and persistent storage different?
- Why does a shorter history help capacity but risk losing meaning?
- Why is a citation or approval ID useful alongside a summary?
- Does selecting relevant passages change the model's context limit?

**Next:** retrieve suitable evidence from a collection instead of manually selecting a passage.

**Reading:** [Context engineering](https://www.anthropic.com/engineering/effective-context-engineering-for-ai-agents).